# 训练过程、AutoFuse OFF/ON 与结果查看

本节在已经完成环境准备的基础上，使用当前章节 `./src` 中已有的训练脚本运行 Qwen3-1.7B SFT AutoFuse 对比任务。学习重点是理解：一次对比任务实际跑了什么、AutoFuse OFF/ON 表示什么、结果保存在哪里、最终应该看哪个指标。

本节学习大纲如下：

- 运行 AutoFuse OFF/ON 对比训练
- 理解四次任务的作用
- 查看结果目录和报告
- 判断训练结果
- 使用 profiling 辅助分析

## 1. 运行 AutoFuse OFF/ON 对比训练

真正启动对比任务的入口是 `run_compare.sh`。这一条命令会串行完成 AutoFuse OFF/ON 的短训练和 profiling。

关键参数可以宏观理解为：

- `--cann-root`：指定当前使用的 CANN。
- `--model`：指定本地 Qwen3-1.7B 模型目录。
- `--data`：指定 Wordle parquet 数据文件。
- `--devices 0,1`：使用两张 NPU。
- `--output-dir`：指定结果统一写入 `tmp/results`。

下面的命令保持 `src/run_compare.sh` 训练入口不变，模型、数据、依赖和训练结果统一使用 `tmp` 下的路径。

In [ ]:
%%bash
export CANN_ROOT=/home/developer/Ascend/ascend-toolkit/latest
source ./tmp/third_party/venv/bin/activate
source "$CANN_ROOT/set_env.sh"
export WORDLE_PARQUET="$PWD/tmp/assets/data/wordle/train-00000-of-00001.parquet"
export TORCHTITAN_NPU_DIR="$PWD/tmp/third_party/torchtitan-npu"
export NGPU=2

bash ./src/run_compare.sh \
  --cann-root "$CANN_ROOT" \
  --model "$PWD/tmp/assets/hf/Qwen3-1.7B" \
  --data "$WORDLE_PARQUET" \
  --devices 0,1 \
  --output-dir "$PWD/tmp/results"

训练全部完成所需时间大约十分钟。

## 2. 理解四次任务的作用

在开始训练前，需要先理解本实践中比较的核心对象：**AutoFuse（自动融合）**。

在深度学习模型执行过程中，一个完整计算流程通常由多个算子组成。不同算子之间可能产生中间数据传输和调度开销，影响模型执行效率。AutoFuse 的主要作用是自动分析计算图中的算子关系，将满足条件的多个算子进行融合，减少算子数量以及中间数据搬运，从而优化模型执行过程。

因此，本实践通过对比 **AutoFuse OFF（关闭 AutoFuse）** 和 **AutoFuse ON（开启 AutoFuse）** 两种状态，观察 AutoFuse 对 Qwen3-1.7B SFT 训练性能的影响。

`run_compare.sh` 会按照固定顺序串行执行四次任务：
- `AutoFuse OFF 训练`
   
   关闭 AutoFuse 功能，运行 10 steps 训练任务。
   该结果作为基准，用于表示未进行融合优化时模型的训练性能。

- `AutoFuse ON 训练`
   
   开启 AutoFuse 功能，运行 10 steps 训练任务。
   通过与 AutoFuse OFF 训练结果对比，可以观察开启 AutoFuse 后训练耗时是否降低。

- `AutoFuse OFF profiling`
   
   关闭 AutoFuse，运行 12 steps，并采集 profiling 信息。
   profiling 用于进一步分析模型执行过程中不同阶段的耗时变化，例如计算、通信以及空闲时间等。

- `AutoFuse ON profiling`
   
   开启 AutoFuse，运行 12 steps，并采集 profiling 信息。
   通过与 AutoFuse OFF profiling 对比，可以辅助分析 AutoFuse 对执行过程产生影响的具体阶段。

训练性能报告主要使用训练阶段 **steps 6-10 的时间均值** 作为最终结果。前几个 step 可以理解为训练启动和运行预热阶段，可能包含额外初始化开销，因此不会直接使用全部 10 个 step 计算平均性能。
本次训练使用固定随机种子：
- 训练阶段使用 seed `42`；
- profiling 阶段使用 seed `43`。
同时，四次任务使用独立缓存环境，减少不同运行之间的相互影响，使 AutoFuse OFF 和 AutoFuse ON 的结果更加具有可比性。

## 3. 查看结果目录和报告

每次运行都会在 `tmp/results` 下创建一个独立目录，目录形式为：

```text
tmp/results/<UTC timestamp>-<pid>-<attempt>/
├── af_off.log
├── af_on.log
├── profiling_off.log
├── profiling_on.log
├── profiling_off/
├── profiling_on/
└── report.md
```

初学者不需要一开始分析所有文件。建议按下面顺序查看：

1. 先打开 `report.md`，看整体结论。
2. 如果训练失败，再查看 `af_off.log` 或 `af_on.log`。
3. 如果需要解释性能阶段变化，再查看 profiling 结果。

`report.md` 是本实践最重要的汇总文件。

## 4. 判断训练结果

训练完成后，可以通过输出目录中的 `report.md` 查看 AutoFuse OFF 和 AutoFuse ON 的对比结果。

首先需要关注训练部分的状态信息：

- `AF_ON_FASTER`

  表示开启 AutoFuse 后，AF ON 的平均训练耗时低于 AF OFF。
  说明在当前短训练条件下，开启 AutoFuse 后模型执行速度更快，观察到了性能提升。

- `NO_GAIN`

  表示 AutoFuse ON 的平均训练耗时没有低于 AutoFuse OFF。
  说明在当前模型、数据规模、训练步数和运行环境下，没有观察到明显的性能提升。但这并不代表 AutoFuse 在所有场景中都没有收益，实际效果仍需要结合模型结构、输入规模和更长时间的训练进行验证。

- `RUN_FAILED`

  表示训练任务执行失败，或者用于性能比较的稳定 step 数据无效。
  出现该状态时，不能直接比较 AutoFuse OFF 和 AutoFuse ON 的性能，需要优先查看训练日志，定位失败原因。


判断 AutoFuse 是否带来性能提升时，主要比较 AutoFuse OFF 和 AutoFuse ON 两种状态下的平均训练耗时。如果 AutoFuse ON 的平均耗时低于 AutoFuse OFF，说明开启 AutoFuse 后本次实验中获得了性能提升；如果 AutoFuse ON 耗时没有降低，则表示当前实验条件下没有观察到明显性能收益。



## 5. 使用 profiling 辅助分析

Profiling 的作用是性能采集与过程分析，用于解释计算、通信等阶段耗时变化。

Profiling 状态可能出现：

- `COMPARABLE`：AutoFuse OFF/ON 两侧 profiling 满足比较条件。
- `UNAVAILABLE`：profiling 失败或结果缺失。
- `INCOMPARABLE`：有结果，但不满足配对比较条件。

当结果为 `COMPARABLE` 时，可以关注以下阶段，单位为 `us`：

- `Computing`：计算耗时；
- `Communication(Not Overlapped)`：没有被计算覆盖的通信耗时；
- `Overlapped`：计算和通信重叠的时间；
- `Free`：空闲耗时；
- `Stage`：完整阶段耗时。

其中 `Overlapped` 表示时间重叠关系，不能简单与其他阶段直接相加。

为了更直观地比较 AutoFuse OFF 和 AutoFuse ON 的阶段性能，报告中还会给出以下计算收益和调度收益指标：

- `Computing 收益`：比较 AutoFuse OFF/ON 的 `Computing` 平均耗时，用于观察开启 AutoFuse 后计算阶段的耗时是否降低。计算方式为 `(OFF Computing - ON Computing) / OFF Computing × 100%`，结果为正值表示 AutoFuse ON 的计算耗时更低。
- `Stage Total 收益`：比较 AutoFuse OFF/ON 的 `Stage` 平均耗时，用于观察完整阶段的总体耗时变化。计算方式为 `(OFF Stage - ON Stage) / OFF Stage × 100%`，结果为正值表示 AutoFuse ON 的完整阶段耗时更低。
- `Stage - Computing 收益`：先分别计算 AutoFuse OFF/ON 的 `Stage - Computing`，再比较两者的差异，用于辅助观察完整阶段中除 `Computing` 之外部分的耗时变化。

以上三个收益均以 AutoFuse OFF 作为基准，收益为正表示 AutoFuse ON 对应耗时降低，收益为负则表示对应耗时增加。

## 6. 课后练习

1. （判断题）本实践中，训练墙钟是判断 AutoFuse ON 是否更快的主要证据。

2. （单选题）`AF_ON_FASTER` 表示什么？
   A. AutoFuse ON 平均耗时更低
   B. AutoFuse ON 一定能在所有模型中获得收益
   C. profiling 一定失败
   D. 模型没有执行训练

3. （多选题）运行`run_compare.sh` 脚本后会串行运行哪些任务？
   A. AutoFuse OFF 训练
   B. AutoFuse ON 训练
   C. AutoFuse OFF profiling
   D. AutoFuse ON profiling

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.03_answer.txt